# khive as a database — first tour

khive treated the way you treat Postgres: a daemon owns the store and is the
single writer; this notebook is a client. Everything here runs against a
**scratch database** in `/tmp` — production stores are never touched.

Wire: Unix socket, length-prefixed JSON frames, protocol-version handshake.
No MCP anywhere in this path.

In [ ]:
import subprocess, os, time, tempfile, pathlib

# Boot a scratch daemon. Short /tmp path on purpose: macOS caps unix-socket
# paths at ~104 bytes and the derived events socket adds ".events.sock".
ROOT = pathlib.Path(tempfile.mkdtemp(prefix="khived-nb-", dir="/tmp"))
(ROOT / "khive.toml").write_text("")   # empty config: no user backends
env = os.environ | {"KHIVE_SOCKET": str(ROOT/"khived.sock"), "KHIVE_PID": str(ROOT/"khived.pid")}
daemon = subprocess.Popen(
    ["kkernel", "mcp", "--daemon", "--config", str(ROOT/"khive.toml"), "--db", str(ROOT/"scratch.db")],
    env=env, cwd=ROOT, stdout=subprocess.DEVNULL, stderr=(ROOT/"daemon.stderr").open("wb"),
)
ROOT

In [ ]:
from khive import Khive, Entity, Note, op

db = Khive(socket_path=str(ROOT/"khived.sock"), timeout=10)

# Readiness = first successful dispatch (the events lane comes up just after
# the main socket).
for _ in range(100):
    try:
        db.stats(); break
    except Exception:
        time.sleep(0.2)
db.session.handshake()[:40], db.stats()

## Substrate: entities, edges, notes

Ids are minted server-side. The edge ontology is closed (17 relations) —
an invalid relation or kind is an error, never coerced.

In [ ]:
lora  = db.entities.create(kind="concept", name="LoRA", description="Low-rank adaptation")
peft  = db.entities.create(kind="concept", name="PEFT")
paper = db.entities.create(kind="document", name="LoRA paper",
                           properties={"year": 2021, "arxiv": "2106.09685"})

# An edge's kind IS its relation — same discriminator slot as every record.
db.graph.link(lora.id, peft.id, "instance_of")
edge = db.graph.link(lora.id, paper.id, "introduced_by", weight=1.0)
db.notes.create(subject="LoRA mechanism",
                content="LoRA freezes base weights; trains rank-r deltas")

edge.kind, edge.namespace, edge.updated_at, db.stats()

In [ ]:
db.graph.neighbors(lora.id)

In [ ]:
db.query("MATCH (c:concept)-[:introduced_by]->(d:document) RETURN c, d")

In [ ]:
# Hybrid search (FTS + vectors, RRF fusion)
db.search("low rank adaptation of large models", limit=5)

## The DB plane: what we are actually here to optimize

`diagnostics()` is `db_diagnostics` — writer acquisitions, busy timeouts,
WAL size, checkpoint counters. `metrics()` reads server-side gauges straight
off the frame without dispatching anything. Counters are per-process
(they reset when the daemon restarts).

In [ ]:
d = db.diagnostics()
{k: d[k] for k in list(d)[:12]}

In [ ]:
db.metrics()

## Writer-pressure demo

N threads bulk-writing through the one daemon writer. Watch acquisitions and
busy timeouts move between two diagnostics reads. This is the harness shape
the real bench module will grow from.

In [ ]:
import threading, time as _t

before = db.diagnostics()
t0 = _t.perf_counter()

def writer(worker: int, count: int = 25):
    w = Khive(socket_path=str(ROOT/"khived.sock"))
    for i in range(count):
        w.entities.create(kind="concept", name=f"bulk-{worker}-{i}")

threads = [threading.Thread(target=writer, args=(w,)) for w in range(8)]
[t.start() for t in threads]; [t.join() for t in threads]

elapsed = _t.perf_counter() - t0
after = db.diagnostics()
print(f"200 writes across 8 clients in {elapsed:.2f}s ({200/elapsed:.0f} writes/s)")
{"stats": db.stats()}

In [ ]:
# What moved, writer-side?
def delta(a, b, keys):
    out = {}
    for k in keys:
        if isinstance(a.get(k), (int, float)) and isinstance(b.get(k), (int, float)):
            out[k] = (a[k], b[k], b[k]-a[k])
    return out
shared = [k for k in after if k in before]
{k: v for k, v in delta(before, after, shared).items() if v[2] != 0}

## Cleanup

In [ ]:
daemon.terminate(); daemon.wait(timeout=10)
import shutil; shutil.rmtree(ROOT, ignore_errors=True)
"scratch daemon stopped, scratch store removed" 